# Create LPDP RISPRO Awards

Creates OpenAlex award rows from Lembaga Pengelola Dana Pendidikan (LPDP) RISPRO research grantee data published on the official LPDP site.

**Prerequisites:**
- Admin/human with AWS credentials runs `scripts/local/lpdp_rispro_to_s3.py` to download the official Gatsby page-data JSON, write parquet, and upload it.
- Contractor has prepared the script, notebook, registry entry, tracker row, and local QA, but does not have S3/Databricks access.
- The downloader writes all parquet columns as strings per `plans/awards/how-to-add-a-funder.md`; this notebook does type conversion with `TRY_CAST` / `TRY_TO_DATE`.

**Data source:** `https://lpdp.kemenkeu.go.id/page-data/grantees/daftar-grantees/page-data.json`  
**Public source page:** `https://lpdp.kemenkeu.go.id/grantees/daftar-grantees/`  
**S3 location:** `s3a://openalex-ingest/awards/lpdp_rispro/lpdp_rispro_grantees.parquet`  
**Local full-source validation on 2026-05-27:** 713 RISPRO research grantee/project rows, 713 unique source-hash native IDs, 100% coverage for title, description, head of research, institution, research focus, RISPRO category, contract year, and landing page; years 2013-2020.

**LPDP funder:**
- funder_id: 4320328515
- display_name: "Lembaga Pengelola Dana Pendidikan"
- country: ID
- alternate title in OpenAlex: "LPDP"

**Source-shape note:** The visible SSR page renders the first 10 cards, but the official Gatsby `page-data.json` contains the full 713-row `granteeListOfGrantees.grantees` array. This notebook uses the first-party page-data JSON, not scraped visible pagination.

**Amount decision:** The source publishes project title, head of research, institution, research focus, RISPRO category, and contract year, but no exact per-project amount or currency. OpenAlex `amount`/`currency` are intentionally NULL with a section 6.7 waiver.

**Mapping summary:** One OpenAlex award row per RISPRO grantee project. Native key is `lpdp-rispro-{year}-{title_slug}-{source_hash}`. `display_name` is `proposal_title`; `description` is a compact source-field summary. `lead_investigator` is the named head of research, with affiliation from the source institution; `affiliation.country` is hardcoded to `ID` because the official RISPRO source is an Indonesian LPDP programme and publishes no per-institution country field. `funding_type='research'`; `funder_scheme` is the RISPRO category.

## Step 1: Create Raw Table

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.lpdp_rispro_raw
USING delta
AS
SELECT
    *,
    current_timestamp() as databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/lpdp_rispro/lpdp_rispro_grantees.parquet`;

In [ ]:
%sql
-- Check raw row count. Local validation on 2026-05-27 found 713 rows.
SELECT COUNT(*) as total_lpdp_rispro_grantees
FROM openalex.awards.lpdp_rispro_raw;

In [ ]:
%sql
-- Verify actual column names before transform logic references them.
DESCRIBE openalex.awards.lpdp_rispro_raw;

In [ ]:
%sql
-- Sample raw LPDP RISPRO data.
SELECT
    funder_award_id,
    source_row_hash,
    display_name,
    head_of_research_name,
    institution,
    research_focus,
    rispro_category,
    contract_year,
    amount,
    currency,
    landing_page_url
FROM openalex.awards.lpdp_rispro_raw
LIMIT 10;

## Step 1.5: Inspect Raw Data, Money, Dates, and Native Keys

In [ ]:
%sql
-- Money-field scan per runbook Step 1.5. This source does not publish per-project amounts.
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'lpdp_rispro_raw'
  AND LOWER(column_name) RLIKE
    'amount|amt|total|value|sum|funded|funding|cost|budget|grant|award|currency|rupiah|idr';

In [ ]:
%sql
-- Confirm amount/date coverage before mapping.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_with_amount,
    COUNT(currency) AS rows_with_currency,
    COUNT(start_date) AS rows_with_start_date,
    MIN(TRY_CAST(start_year AS INT)) AS min_year,
    MAX(TRY_CAST(start_year AS INT)) AS max_year
FROM openalex.awards.lpdp_rispro_raw;

In [ ]:
%sql
-- Native-key inspection. Source row hash is derived from year/category/title/PI/institution/focus.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT source_row_hash) AS distinct_source_row_hashes,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT display_name) AS distinct_project_titles,
    COUNT(DISTINCT head_of_research_name) AS distinct_heads_of_research
FROM openalex.awards.lpdp_rispro_raw;

In [ ]:
%sql
-- RISPRO category and year distribution.
SELECT
    contract_year,
    rispro_category,
    COUNT(*) AS cnt
FROM openalex.awards.lpdp_rispro_raw
GROUP BY contract_year, rispro_category
ORDER BY TRY_CAST(contract_year AS INT), cnt DESC;

In [ ]:
%sql
-- Research-focus distribution.
SELECT research_focus, COUNT(*) AS cnt
FROM openalex.awards.lpdp_rispro_raw
GROUP BY research_focus
ORDER BY cnt DESC, research_focus
LIMIT 50;

In [ ]:
%sql
-- Document the amount decision that will be applied in Step 2.
SELECT
    amount_decision,
    page_data_url,
    source_page_url,
    COUNT(*) AS rows
FROM openalex.awards.lpdp_rispro_raw
GROUP BY amount_decision, page_data_url, source_page_url;

## Step 1.6: Funder Existence Fail-Fast

In [ ]:
%sql
-- Must return exactly 1 row. If this is empty, STOP: the funder is missing from openalex.common.funder.
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320328515;

## Step 2: Transform to OpenAlex Award Schema

In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.lpdp_rispro_awards
USING delta
AS
WITH
lpdp_funder AS (
    SELECT
        funder_id,
        display_name,
        ror_id,
        doi
    FROM openalex.common.funder
    WHERE funder_id = 4320328515
),

raw_prepared AS (
    SELECT
        *,
        LOWER(TRIM(funder_award_id)) AS native_award_id,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS parsed_start_date,
        TRY_CAST(start_year AS INT) AS parsed_start_year
    FROM openalex.awards.lpdp_rispro_raw
    WHERE funder_award_id IS NOT NULL
      AND TRIM(funder_award_id) != ''
      AND display_name IS NOT NULL
      AND TRIM(display_name) != ''
),

awards_transformed AS (
    SELECT
        abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000 as id,

        TRIM(g.display_name) as display_name,

        CASE
            WHEN g.description IS NULL OR TRIM(g.description) = '' THEN NULL
            ELSE TRIM(g.description)
        END as description,

        f.funder_id,
        g.native_award_id as funder_award_id,

        CAST(NULL AS DOUBLE) as amount,
        CAST(NULL AS STRING) as currency,

        struct(
            CONCAT('https://openalex.org/F', f.funder_id) as id,
            f.display_name,
            f.ror_id,
            f.doi
        ) as funder,

        'research' as funding_type,

        NULLIF(TRIM(g.funder_scheme), '') as funder_scheme,

        'lpdp_rispro_grantees' as provenance,

        g.parsed_start_date as start_date,
        CAST(NULL AS DATE) as end_date,
        COALESCE(YEAR(g.parsed_start_date), g.parsed_start_year) as start_year,
        CAST(NULL AS INT) as end_year,

        struct(
            NULLIF(TRIM(g.lead_investigator_given_name), '') as given_name,
            NULLIF(TRIM(g.lead_investigator_family_name), '') as family_name,
            CAST(NULL AS STRING) as orcid,
            g.parsed_start_date as role_start,
            struct(
                NULLIF(TRIM(g.institution), '') as name,
                'ID' as country,
                CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) as ids
            ) as affiliation
        ) as lead_investigator,

        CAST(NULL AS STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >) as co_lead_investigator,

        CAST(NULL AS ARRAY<STRUCT<
            given_name:STRING,
            family_name:STRING,
            orcid:STRING,
            role_start:DATE,
            affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
        >>) as investigators,

        NULLIF(TRIM(g.landing_page_url), '') as landing_page_url,
        CAST(NULL AS STRING) as doi,

        concat('https://api.openalex.org/works?filter=awards.id:G', abs(xxhash64(CONCAT(f.funder_id, ':', g.native_award_id))) % 9000000000) as works_api_url,

        current_timestamp() as created_date,
        current_timestamp() as updated_date

    FROM raw_prepared g
    CROSS JOIN lpdp_funder f
)

SELECT * FROM awards_transformed;

## Step 3: Delete Previous Source Rows and Insert Priority 136

In [ ]:
%sql
-- Remove previous LPDP RISPRO data before inserting fresh data.
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'lpdp_rispro_grantees' AND priority = 136;

-- Insert into openalex_awards_raw with exact OpenAlex awards schema plus priority.
INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id,
    display_name,
    description,
    funder_id,
    funder_award_id,
    amount,
    currency,
    funder,
    funding_type,
    funder_scheme,
    provenance,
    start_date,
    end_date,
    start_year,
    end_year,
    lead_investigator,
    co_lead_investigator,
    investigators,
    landing_page_url,
    doi,
    works_api_url,
    created_date,
    updated_date,
    136 as priority
FROM openalex.awards.lpdp_rispro_awards;

## Handoff/Admin Notes

Contractor-complete handoff stops here. Admin must upload the parquet to S3, run this notebook in Databricks, inspect the Step 6 verification cells, and only then decide whether to add scheduled job YAML. No job YAML is included in this PR.

## Step 6: Verification Queries

In [ ]:
%sql
SELECT COUNT(*) as total_lpdp_rispro_awards
FROM openalex.awards.lpdp_rispro_awards;

In [ ]:
%sql
-- Confirm the transformed table has the OpenAlex awards columns only.
DESCRIBE openalex.awards.lpdp_rispro_awards;

In [ ]:
%sql
-- Sample transformed awards.
SELECT
    id,
    display_name,
    funder_award_id,
    funder_scheme,
    funding_type,
    amount,
    currency,
    start_date,
    end_date,
    lead_investigator.given_name AS pi_given_name,
    lead_investigator.family_name AS pi_family_name,
    lead_investigator.affiliation.name AS institution,
    lead_investigator.affiliation.country AS institution_country,
    landing_page_url
FROM openalex.awards.lpdp_rispro_awards
LIMIT 10;

In [ ]:
%sql
-- Check ID and native-key uniqueness.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT id) AS distinct_openalex_ids,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids
FROM openalex.awards.lpdp_rispro_awards;

In [ ]:
%sql
-- Check funder distribution. Should be only LPDP F4320328515.
SELECT funder.display_name, funder_id, COUNT(*) as cnt
FROM openalex.awards.lpdp_rispro_awards
GROUP BY funder.display_name, funder_id
ORDER BY cnt DESC;

In [ ]:
%sql
-- Data completeness.
SELECT
    COUNT(*) as total,
    COUNT(display_name) as has_title,
    COUNT(description) as has_description,
    COUNT(amount) as has_amount,
    COUNT(start_date) as has_start_date,
    COUNT(end_date) as has_end_date,
    COUNT(landing_page_url) as has_url,
    COUNT(lead_investigator.family_name) as has_pi_family,
    COUNT(lead_investigator.affiliation.name) as has_institution,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) as pct_with_amount,
    ROUND(try_divide(COUNT(description), COUNT(*)) * 100.0, 1) as pct_description,
    ROUND(try_divide(COUNT(lead_investigator.family_name), COUNT(*)) * 100.0, 1) as pct_pi_family,
    ROUND(try_divide(COUNT(lead_investigator.affiliation.name), COUNT(*)) * 100.0, 1) as pct_institution
FROM openalex.awards.lpdp_rispro_awards;

In [ ]:
%sql
-- Amount/currency waiver check. LPDP page-data does not publish exact per-project budgets.
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS rows_with_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    COUNT(DISTINCT currency) AS distinct_currencies,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    AVG(amount) AS avg_amount
FROM openalex.awards.lpdp_rispro_awards;

In [ ]:
%sql
-- Year and scheme distribution.
SELECT start_year, funder_scheme, COUNT(*) as cnt
FROM openalex.awards.lpdp_rispro_awards
WHERE start_year IS NOT NULL
GROUP BY start_year, funder_scheme
ORDER BY start_year, cnt DESC;

In [ ]:
%sql
-- Institution distribution.
SELECT lead_investigator.affiliation.name AS institution, COUNT(*) AS cnt
FROM openalex.awards.lpdp_rispro_awards
GROUP BY lead_investigator.affiliation.name
ORDER BY cnt DESC, institution
LIMIT 50;

In [ ]:
%sql
-- Verify rows inserted into the raw awards table at priority 136.
SELECT
    priority,
    provenance,
    COUNT(*) as cnt,
    COUNT(DISTINCT funder_award_id) as distinct_funder_award_ids,
    COUNT(amount) as rows_with_amount
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'lpdp_rispro_grantees' AND priority = 136
GROUP BY priority, provenance;